# Project Setup

## Creating Folders

In [ ]:
!echo "Creating data folder"
!mkdir -p data/bronze data/silver data/gold

## Downloading datasets

In [ ]:


!echo "Downloading dataset"
!curl -L -o data/saltosp-house-dataset-2022.zip https://www.kaggle.com/api/v1/datasets/download/marcosangelo/saltosp-house-dataset-2022

!echo "Unzipping dataset"
!unzip data/saltosp-house-dataset-2022.zip -d data/bronze

## Dataset Salto (São Paulo)

- concat spreadsheets
- add column type (tipo)
- split price value into buy/sell value or rent value (per month)
- parse fiels to correct type
- remove / alter sensitive info
- 


In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv("data/bronze/real_estate_salto_sp_apartamento_aluguel_5.csv")
df2 = pd.read_csv("data/bronze/real_estate_salto_sp_apartamento_venda_35.csv")
df_ap = pd.concat([df1, df2], ignore_index=True)
df_ap["tipo"] = "apartamento"

df3 = pd.read_csv("data/bronze/real_estate_salto_sp_casa_aluguel_3.csv")
df4 = pd.read_csv("data/bronze/real_estate_salto_sp_casa_venda_35.csv")
df_hs = pd.concat([df3, df4], ignore_index=True)
df_hs["tipo"] = "casa"

df_full = pd.concat([df_ap, df_hs], ignore_index=True)

df_full.to_csv("data/silver/real_estate_salto_sp.csv")

df_full.info()

In [ ]:
print(df_full.isnull().sum())

print(df_full.isna().sum())

In [ ]:
df_full.head()

## Parse price and operation type

In [ ]:
df_full["operacao"] = df_full["preco_casa"] \
    .str.contains(r"/m[êe]s", case=False, na=False) \
    .map({True: "aluguel", False: "venda"})

df_full["valor"] = (
    df_full["preco_casa"]
    .str.replace(r"[^0-9]+", "", regex=True)
    .astype(float)
)

df_full.drop(columns=["preco_casa"], inplace=True)

df_full

In [ ]:
import re

def parse_metros(valor: str) -> float:

    s = str(valor).strip()

    # empty or invalues values 
    if s == "" or not re.match(r"\d+", s):
        return 0

    if "-" in s:
        a, b = s.split("-", 1)
        if a == "" or b == "":
            print(valor)
        return float((float(a) + float(b)) / 2)

    return float(s)

In [ ]:
df_full["metros"] = df_full["metros_casa"].map(parse_metros)
df_full["quartos"] = df_full["quartos_casa"].map(parse_metros)
df_full["banheiros"] = df_full["banheiros_casa"].map(parse_metros)
df_full["vagas"] = df_full["vagas_casa"].map(parse_metros)

df_full.drop(columns=["metros_casa", "quartos_casa", "banheiros_casa", "vagas_casa"], inplace=True)

df_full

In [ ]:
df_full.info()

In [ ]:
df_full.to_csv("data/silver/real_estate_salto_sp.csv", index=False)